# Semantic Document Search

**Guide:** [`070-semantic-document-search.md`](../guides/concepts/070-semantic-document-search.md)

Semantic search allows computers to understand the meaning behind user queries rather than relying only on exact keyword matching. Using FAISS (Facebook AI Similarity Search), we can build a high-performance system that searches through hundreds or even thousands of documents by meaning and not just by text overlap. This approach enables smarter, faster and more context-aware information retrieval.

## Implementation
- Document Ingestion: PDF, DOCX and TXT files are read and converted into text.
- Chunking: Each document is split into smaller, meaningful parts.
- Embedding Generation: Text chunks are converted into dense vector representations.
- FAISS Indexing: Embeddings are stored in FAISS for efficient similarity search.
- Query Encoding: A user query is embedded into the same vector space.
- Similarity Search: FAISS finds top matches based on cosine similarity.
- Results Display: The most relevant document snippets are shown.

## Step 1: Install Dependencies
We need to install the required dependencies such as faiss-cpu, sentence-transformers, python-docx.

Install the notebook dependencies from the repository root before running this lesson:

```powershell
python -m pip install -e ".[notebooks]"
```

The first embedding run downloads `all-MiniLM-L6-v2` from Hugging Face. It needs network access once; later runs use the local model cache.

## Step 2: Import Libraries
We will import the necessary libraires such as os, docx, numpy, SentenceTransformers, faiss.

In [ ]:
from pathlib import Path

import faiss
from docx import Document
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

## Step 3: Extract Text from Documents
We need to define the function for document loading,

- Reads different file formats like .pdf, .docx and .txt.
- Ensures content is extracted as plain text for embedding generation.

In [ ]:
def extract_text_from_file(file_path):

    text = ""
    file_path = Path(file_path)
    suffix = file_path.suffix.lower()
    if suffix == ".pdf":
        reader = PdfReader(file_path)
        for page in reader.pages:
            text += (page.extract_text() or "") + "\n"
    elif suffix == ".docx":
        doc = Document(file_path)
        for para in doc.paragraphs:
            text += para.text + "\n"
    elif suffix == ".txt":
        text = file_path.read_text(encoding="utf-8")
    return text.strip()

## Step 4: Split Text into Chunks
- Divides long documents into smaller segments (chunks).
- Improves search accuracy and performance by focusing on smaller text units.

In [ ]:
def chunk_text(text, chunk_size=300):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    words = text.split()
    return [
        " ".join(words[start:start + chunk_size])
        for start in range(0, len(words), chunk_size)
        if words[start:start + chunk_size]
    ]

## Step 5: Load and Process Documents
Here:
- Reads all files in the documents/ folder.
- Splits them into manageable text chunks and stores the source file name for each.

In [ ]:
folder_path = Path.cwd() / "documents"
if not folder_path.is_dir():
    folder_path = Path.cwd().parent / "documents"
if not folder_path.is_dir():
    raise FileNotFoundError("Could not find the repository documents/ folder")
documents = []
doc_sources = []

for path in sorted(folder_path.iterdir()):
    if path.suffix.lower() in {".pdf", ".docx", ".txt"}:
        print(f"Reading file: {path.name}")
        content = extract_text_from_file(path)
        chunks = chunk_text(content)
        documents.extend(chunks)
        doc_sources.extend([path.name] * len(chunks))

if not documents:
    raise ValueError(f"No readable text documents found in {folder_path}")
print(f"\nLoaded {len(documents)} text chunks from {folder_path}.")

## Step 6: Generate Text Embeddings
Here we will:
- Converts text chunks into vector representations using SentenceTransformer.
- Normalizes vectors for cosine similarity in FAISS.
- Shows embedding progress for transparency.

In [ ]:
if not documents:
    raise ValueError("No documents to embed")

model = SentenceTransformer("all-MiniLM-L6-v2")
print("\nGenerating embeddings... (this may take a minute)")

embeddings = model.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True,
).astype("float32")

faiss.normalize_L2(embeddings)
print(f"Embeddings shape: {embeddings.shape}")

## Step 7: Create FAISS Index
- Initializes a FAISS IndexFlatIP index (for cosine similarity).
- Adds all text embeddings into the FAISS index for fast retrieval.

In [ ]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
print(f"FAISS index created with {index.ntotal} vectors.")

## Step 8: Define Cleaning and Search Functions
1. clean_text(): Removes unwanted formatting and extra spaces.
2. semantic_search_best():
- Converts the query into a vector.
- Searches the FAISS index for similar embeddings.
- Displays the best matches with readable snippets.

In [ ]:
import re
import textwrap


def clean_text(text):

    text = re.sub(r'[#=*`~_-]+', '', text)
    text = re.sub(r'\*\*(.*?)\*\*', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def semantic_search_best(query, top_k=1, wrap_width=100, similarity_threshold=0.35, snippet_length=300):
    if not query.strip():
        raise ValueError("query must not be empty")
    if index.ntotal == 0:
        raise ValueError("index has no document vectors")
    top_k = min(max(1, top_k), index.ntotal)

    query_embedding = model.encode([query]).astype('float32')
    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, top_k)

    print("\nTop Semantic Search Result(s):")
    print("=" * 120)

    results_shown = 0

    for rank, idx in enumerate(I[0]):
        if idx < 0:
            continue
        score = D[0][rank]
        if score < similarity_threshold:
            continue

        snippet = clean_text(documents[idx])[:snippet_length]
        wrapped_snippet = textwrap.fill(snippet, width=wrap_width)

        print(f"\nRank {rank + 1}")
        print(f"Source File     : {doc_sources[idx]}")
        print(f"Similarity Score: {score:.4f}")
        print("-" * 120)
        print(f"Preview Snippet:\n{wrapped_snippet}")
        print("=" * 120)
        results_shown += 1

    if results_shown == 0:
        print("No strong semantic matches found for your query.")

## Step 9: Run Semantic Search
- Retrieves top semantically relevant chunks for each query.
- Displays source document name, similarity score and wrapped text preview.

In [ ]:
semantic_search_best("applications of artificial intelligence")

In [ ]:
semantic_search_best("database systems and AI", top_k=3)

## Advantages
- High Performance: FAISS enables instant retrieval even across thousands of vectors.
- Semantic Understanding: Captures meaning beyond keywords.
- Multi-format Support: Works with PDF, DOCX and TXT files.
- Scalable: Easily extensible to large datasets.